# 01. Case Base — Data Acquisition & Preprocessing

Tahap ini digunakan untuk membangun **case base** putusan perkara **perdata waris**.

Sesuai instruksi tugas, tahap ini mencakup:

1. Menyiapkan struktur folder project.
2. Mengambil minimal 30 dokumen putusan dalam format PDF.
3. Mengonversi PDF menjadi teks.
4. Membersihkan teks dari header, footer, watermark, nomor halaman, dan karakter tidak penting.
5. Menyimpan hasil teks bersih ke folder `data/raw/`.
6. Membuat log validasi pembersihan di folder `logs/`.

> Letakkan file PDF putusan asli di folder `data/pdf/`, bukan di `data/raw/`.


## 1. Instalasi dan Import Library

In [1]:
%pip install pypdf pandas tqdm -q

import os
import re
import shutil
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from pypdf import PdfReader

print("Library siap digunakan.")


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Library siap digunakan.


## 2. Membuat Struktur Folder

Struktur folder mengikuti instruksi tugas:

```text
/data/
  /pdf/        -> tempat PDF asli
  /raw/        -> hasil ekstraksi teks bersih
  /processed/  -> hasil representasi kasus tahap 2
  /eval/       -> query dan evaluasi
  /results/    -> hasil prediksi
/logs/         -> cleaning.log dan cleaning.csv
```

In [2]:
# Gunakan path relatif dari notebook.
# Jika notebook berada di folder /notebooks/, maka root project adalah satu folder di atasnya.
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

DATA_DIR = PROJECT_ROOT / "data"
PDF_DIR = DATA_DIR / "pdf"
RAW_TXT_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = DATA_DIR / "eval"
RESULTS_DIR = DATA_DIR / "results"
LOG_DIR = PROJECT_ROOT / "logs"

FOLDERS = [
    PDF_DIR,
    RAW_TXT_DIR,
    PROCESSED_DIR,
    EVAL_DIR,
    RESULTS_DIR,
    LOG_DIR,
]

for folder in FOLDERS:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("PDF folder   :", PDF_DIR)
print("RAW TXT folder:", RAW_TXT_DIR)
print("LOG folder   :", LOG_DIR)

Project root : /Users/muhammadsenopati/cbr-perdata-waris
PDF folder   : /Users/muhammadsenopati/cbr-perdata-waris/data/pdf
RAW TXT folder: /Users/muhammadsenopati/cbr-perdata-waris/data/raw
LOG folder   : /Users/muhammadsenopati/cbr-perdata-waris/logs


## 3. Mengecek Jumlah PDF

Tugas mensyaratkan minimal **30 dokumen putusan**. File PDF harus dimasukkan ke folder:

```text
data/pdf/
```

Jika masih ada PDF di `data/raw/`, sel berikut dapat memindahkannya otomatis ke `data/pdf/`.


In [3]:
TARGET_PDFS = 30

# Opsional: pindahkan PDF yang salah taruh di data/raw ke data/pdf
pdf_in_raw = sorted(RAW_TXT_DIR.glob("*.pdf"))

if pdf_in_raw:
    print(f"Ditemukan {len(pdf_in_raw)} PDF di data/raw/. File akan dipindahkan ke data/pdf/.")
    for pdf_path in pdf_in_raw:
        destination = PDF_DIR / pdf_path.name
        if not destination.exists():
            shutil.move(str(pdf_path), str(destination))
        else:
            print(f"Skip karena sudah ada di data/pdf/: {pdf_path.name}")

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print(f"Target minimal PDF : {TARGET_PDFS}")
print(f"Jumlah PDF ditemukan: {len(pdf_files)}")

if len(pdf_files) < TARGET_PDFS:
    print(f"PERINGATAN: jumlah PDF masih kurang dari {TARGET_PDFS}. Tambahkan PDF lagi ke data/pdf/.")
else:
    print("Jumlah PDF sudah memenuhi syarat minimal.")

Target minimal PDF : 30
Jumlah PDF ditemukan: 50
Jumlah PDF sudah memenuhi syarat minimal.


## 4. Fungsi Ekstraksi Teks dari PDF

Fungsi ini membaca setiap halaman PDF dan menggabungkan hasil ekstraksi menjadi satu teks utuh.


In [4]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    """Ekstrak teks dari file PDF menggunakan pypdf."""
    text_parts = []

    try:
        reader = PdfReader(str(pdf_path))

        for page in reader.pages:
            page_text = page.extract_text() or ""
            if page_text.strip():
                text_parts.append(page_text)

    except Exception as error:
        print(f"Gagal membaca {pdf_path.name}: {error}")
        return ""

    return "\n".join(text_parts)


print("Fungsi extract_text_from_pdf() siap digunakan.")

Fungsi extract_text_from_pdf() siap digunakan.


## 5. Fungsi Pembersihan Teks

Pembersihan dibuat **tidak terlalu agresif** agar informasi hukum tetap aman, misalnya:

- nomor perkara,
- tanggal,
- pasal,
- nama pihak,
- amar putusan,
- tanda `/`, `.`, `,`, `:`, `;`, `-`, dan tanda kurung.

Yang dihapus hanya bagian yang sering menjadi noise PDF Mahkamah Agung, seperti header, footer, watermark, nomor halaman, dan disclaimer.


In [5]:
def clean_legal_text(text: str) -> str:
    """Membersihkan teks putusan tanpa merusak informasi hukum penting."""
    if not text:
        return ""

    # Hilangkan null character
    text = text.replace("\x00", " ")

    # Pola noise umum pada PDF putusan MA
    noise_patterns = [
        r"direktori\s+putusan\s+mahkamah\s+agung\s+republik\s+indonesia",
        r"putusan\.mahkamahagung\.go\.id",
        r"mahkamah\s+agung\s+republik\s+indonesia",
        r"kepaniteraan\s+mahkamah\s+agung.*",
        r"direktorat\s+jenderal\s+badan\s+peradilan.*",
        r"halaman\s+\d+\s+dari\s+\d+.*",
        r"hal\s+\d+\s+put\s+no.*",
        r"page\s+\d+",
        r"disclaimer.*",
        r"dalam\s+hal\s+anda\s+menemukan\s+incompabilitas.*",
        r"dalam\s+hal\s+anda\s+menemukan\s+inkompatibilitas.*",
    ]

    for pattern in noise_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    # Lowercase untuk normalisasi
    text = text.lower()

    # Pertahankan karakter penting hukum:
    # huruf, angka, spasi, garis miring, titik, koma, titik dua, titik koma, strip, dan tanda kurung
    text = re.sub(r"[^a-z0-9\s\/\(\)\,\.\-\:\;]", " ", text)

    # Normalisasi spasi
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


print("Fungsi clean_legal_text() siap digunakan.")

Fungsi clean_legal_text() siap digunakan.


## 6. Fungsi Validasi Keutuhan Teks

Coverage dihitung berdasarkan jumlah kata:

```text
coverage = jumlah kata setelah dibersihkan / jumlah kata hasil ekstraksi PDF × 100
```

Target validasi dibuat mengikuti instruksi tugas, yaitu minimal **80% isi putusan tersedia**.

Catatan: jika coverage sedikit di bawah 80%, cek manual isi file `.txt`. Bisa jadi PDF memiliki banyak header/footer/disclaimer yang memang harus dibuang.


In [6]:
def calculate_coverage(raw_text: str, cleaned_text: str) -> tuple:
    raw_words = len(raw_text.split())
    cleaned_words = len(cleaned_text.split())

    if raw_words == 0:
        coverage = 0.0
    else:
        coverage = round((cleaned_words / raw_words) * 100, 2)

    return raw_words, cleaned_words, coverage


def check_key_sections(text: str) -> dict:
    """Cek sederhana apakah bagian penting putusan masih tersedia."""
    keywords = {
        "nomor_perkara": bool(re.search(r"nomor|no\.", text, flags=re.IGNORECASE)),
        "menimbang": "menimbang" in text,
        "mengingat": "mengingat" in text,
        "mengadili": "mengadili" in text,
        "pasal": "pasal" in text,
        "putusan": "putusan" in text,
    }

    score = round((sum(keywords.values()) / len(keywords)) * 100, 2)
    keywords["section_score_percent"] = score

    return keywords


print("Fungsi validasi siap digunakan.")

Fungsi validasi siap digunakan.


## 7. Proses Konversi PDF ke TXT dan Pembersihan

Output utama tahap ini adalah:

```text
data/raw/case_001.txt
data/raw/case_002.txt
...
logs/cleaning.log
logs/cleaning.csv
```


In [7]:
LOG_TXT_PATH = LOG_DIR / "cleaning.log"
LOG_CSV_PATH = LOG_DIR / "cleaning.csv"

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

cleaning_log = []

print(f"Mulai memproses {len(pdf_files)} file PDF...\n")

for index, pdf_path in enumerate(tqdm(pdf_files, desc="Case Base"), start=1):
    case_id = f"case_{index:03d}"
    txt_path = RAW_TXT_DIR / f"{case_id}.txt"

    raw_text = extract_text_from_pdf(pdf_path)
    cleaned_text = clean_legal_text(raw_text)

    raw_words, cleaned_words, coverage = calculate_coverage(raw_text, cleaned_text)
    section_check = check_key_sections(cleaned_text)

    # Valid jika coverage >= 80% atau section penting masih cukup lengkap
    # Untuk tampilan tugas, tetap pakai coverage sebagai patokan utama.
    status = "VALID" if coverage >= 80 else "PERLU CEK"

    with open(txt_path, "w", encoding="utf-8") as file:
        file.write(cleaned_text)

    log_row = {
        "case_id": case_id,
        "pdf_file": pdf_path.name,
        "txt_file": txt_path.name,
        "raw_words": raw_words,
        "cleaned_words": cleaned_words,
        "coverage_percent": coverage,
        "status": status,
        **section_check,
    }

    cleaning_log.append(log_row)

    print(f"{status}: {pdf_path.name} -> {txt_path.name} | coverage={coverage}% | section_score={section_check['section_score_percent']}%")

log_df = pd.DataFrame(cleaning_log)

# Simpan CSV
log_df.to_csv(LOG_CSV_PATH, index=False)

# Simpan log TXT
with open(LOG_TXT_PATH, "w", encoding="utf-8") as log_file:
    log_file.write("=== LOG FILE PEMBERSIHAN DATA PUTUSAN ===\n")
    log_file.write("Tahap 1: Case Base - Data Acquisition & Preprocessing\n\n")
    log_file.write(log_df.to_string(index=False))

print("\nOutput tahap 1 selesai.")
print(f"TXT folder  : {RAW_TXT_DIR}")
print(f"Cleaning log: {LOG_TXT_PATH}")
print(f"Cleaning CSV: {LOG_CSV_PATH}")

Mulai memproses 50 file PDF...



Case Base:   2%|▏         | 1/50 [00:00<00:07,  6.42it/s]

VALID: putusan_1042_pk_pdt_2024_20260621101005.pdf -> case_001.txt | coverage=86.26% | section_score=83.33%


Case Base:   4%|▍         | 2/50 [00:00<00:12,  3.75it/s]

VALID: putusan_1065_pk_pdt_2025_20260611205716.pdf -> case_002.txt | coverage=87.69% | section_score=83.33%


...

VALID: putusan_82_k_pdt_2026_20260608122143.pdf -> case_046.txt | coverage=87.16% | section_score=66.67%
VALID: putusan_845_k_pdt_2024_20260621102723.pdf -> case_047.txt | coverage=89.33% | section_score=66.67%


Case Base:  98%|█████████▊| 49/50 [00:13<00:00,  4.65it/s]

VALID: putusan_955_pk_pdt_2025_20260612123753.pdf -> case_048.txt | coverage=86.86% | section_score=66.67%
VALID: putusan_966_pk_pdt_2025_20260612122913.pdf -> case_049.txt | coverage=86.62% | section_score=83.33%


Case Base: 100%|██████████| 50/50 [00:13<00:00,  3.74it/s]

VALID: putusan_976_k_pdt_2026_20260609142647.pdf -> case_050.txt | coverage=87.13% | section_score=50.0%

Output tahap 1 selesai.
TXT folder  : /Users/muhammadsenopati/cbr-perdata-waris/data/raw
Cleaning log: /Users/muhammadsenopati/cbr-perdata-waris/logs/cleaning.log
Cleaning CSV: /Users/muhammadsenopati/cbr-perdata-waris/logs/cleaning.csv


## 8. Ringkasan Validasi

Bagian ini menampilkan jumlah dokumen valid, dokumen yang perlu dicek, dan contoh log pembersihan.


In [8]:
if LOG_CSV_PATH.exists():
    log_df = pd.read_csv(LOG_CSV_PATH)

    total_docs = len(log_df)
    valid_docs = (log_df["status"] == "VALID").sum()
    warning_docs = (log_df["status"] == "PERLU CEK").sum()

    print("=== RINGKASAN VALIDASI TAHAP 1 ===")
    print(f"Total dokumen diproses : {total_docs}")
    print(f"Dokumen valid          : {valid_docs}")
    print(f"Dokumen perlu cek      : {warning_docs}")
    print(f"Rata-rata coverage     : {log_df['coverage_percent'].mean():.2f}%")

    if total_docs >= 30:
        print("Syarat jumlah dokumen minimal 30: TERPENUHI")
    else:
        print("Syarat jumlah dokumen minimal 30: BELUM TERPENUHI")

    display(log_df.head())

else:
    print("File cleaning.csv belum ditemukan.")

=== RINGKASAN VALIDASI TAHAP 1 ===
Total dokumen diproses : 50
Dokumen valid          : 50
Dokumen perlu cek      : 0
Rata-rata coverage     : 87.32%
Syarat jumlah dokumen minimal 30: TERPENUHI


,case_id,pdf_file,txt_file,raw_words,cleaned_words,coverage_percent,status,nomor_perkara,menimbang,mengingat,mengadili,pasal,putusan,section_score_percent
0,case_001,putusan_1042_pk_pdt_2024_20260621101005.pdf,case_001.txt,2736,2360,86.26,VALID,True,True,False,True,True,True,83.33
1,case_002,putusan_1065_pk_pdt_2025_20260611205716.pdf,case_002.txt,6435,5643,87.69,VALID,True,True,False,True,True,True,83.33
2,case_003,putusan_1120_pk_pdt_2025_20260612122607.pdf,case_003.txt,4263,3711,87.05,VALID,True,True,False,True,True,True,83.33
3,case_004,putusan_1354_pk_pdt_2024_20260621102041.pdf,case_004.txt,3195,2778,86.95,VALID,True,True,False,True,False,True,66.67
4,case_005,putusan_1458_pk_pdt_2025_20260602213350.pdf,case_005.txt,3302,2887,87.43,VALID,True,True,False,True,True,True,83.33


## 9. Melihat Contoh Hasil Teks Bersih

Gunakan bagian ini untuk memastikan hasil `.txt` masih memuat isi penting putusan.


In [9]:
txt_files = sorted(RAW_TXT_DIR.glob("case_*.txt"))

print(f"Total file TXT: {len(txt_files)}")

if txt_files:
    sample_path = txt_files[0]

    with open(sample_path, "r", encoding="utf-8") as file:
        sample_text = file.read()

    print(f"Contoh file: {sample_path.name}")
    print("=" * 80)
    print(sample_text[:2000])
else:
    print("Belum ada file TXT di data/raw/.")

Total file TXT: 50
Contoh file: case_001.txt
p u t u s a n nomor 1042 pk/pdt/2024 demi keadilan berdasarkan ketuhanan yang maha esa m a h k a m a h a g u n g memeriksa perkara perdata pada pemeriksaan peninjauan kembali telah memutus sebagai berikut dalam perkara antara: pt bama bumi sentosa, berkedudukan di jalan perak barat nomor 225, kelurahan perak utara, kecamatan pabean cantian, kota surabaya, provinsi jawa timur, diwakili oleh direktur utama, kikin abdul hakim, dalam hal ini memberi kuasa kepada prof. dr. h. sunarno edy wibowo, s.h., m.hum., dan kawan-kawan, para advokat pada kantor advokat wibowo partner, berkantor di jalan rungkut barata xii/32, kota surabaya, provinsi jawa timur , berdasarkan surat kuasa khusus tanggal 22 februari 2023; pemohon peninjauan kembali/tergugat; l a w a n pt mardohar catur tunggal gaya, berkedudukan di jalan perjuangan nomor 77, kebun jeruk, kecamatan kebon jeruk, kota jakarta barat, dki jakarta, 11530; termohon peninjauan kembali/tergugat; mahkama

## 10. Kesimpulan Tahap 1

Tahap 1 menghasilkan case base awal berupa file teks putusan yang sudah dibersihkan.

Output yang dihasilkan:

1. Folder `data/raw/` berisi file `case_001.txt`, `case_002.txt`, dan seterusnya.
2. File `logs/cleaning.log` sebagai catatan proses pembersihan.
3. File `logs/cleaning.csv` sebagai rekap validasi coverage.
4. Data siap digunakan pada tahap berikutnya, yaitu **Case Representation**.
